### 1. Install & Verify Dependencies

In [ ]:
# Install required packages in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Installing dependencies on Google Colab...")
    !pip install -q transformers datasets accelerate scikit-learn seaborn matplotlib
except ImportError:
    IN_COLAB = False
    print("Running locally on Jupyter!")

import torch
print(f"\nPyTorch Version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Notice: No GPU detected. Make sure to set Runtime > Change runtime type > T4 GPU for fast training!")

### 2. Load the Multi-Industry Benchmark Dataset

In [ ]:
import os
import pandas as pd
import numpy as np

# Direct Raw GitHub URL ensures data loads instantly in Colab with no clone issues
GITHUB_DATA_URL = "https://raw.githubusercontent.com/Khitishnayak/Customer-Complaint-Classification/main/data/processed/complaints_colab_sample.csv"
LOCAL_PATH = "../data/processed/complaints_colab_sample.csv"

if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
else:
    print(f"Fetching dataset from GitHub repository...")
    df = pd.read_csv(GITHUB_DATA_URL)

df = df.dropna(subset=["text", "category"]).copy()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 15]

categories = sorted(df["category"].unique())
label2id = {cat: i for i, cat in enumerate(categories)}
id2label = {i: cat for i, cat in enumerate(categories)}
df["label"] = df["category"].map(label2id)

print(f"Successfully loaded {len(df):,} complaints across {len(categories)} categories.")
print("\nCategories:", categories)

### 3. Split Dataset into Train, Validation, and Test Sets

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified Split: 80% Train, 10% Validation, 10% Test
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print(f"Train samples:      {len(train_df):,}")
print(f"Validation samples: {len(val_df):,}")
print(f"Test samples:       {len(test_df):,}")

### 4. Tokenization using Hugging Face AutoTokenizer

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )

train_dataset = Dataset.from_pandas(train_df[["text", "label"]]).map(tokenize_fn, batched=True)
val_dataset = Dataset.from_pandas(val_df[["text", "label"]]).map(tokenize_fn, batched=True)
test_dataset = Dataset.from_pandas(test_df[["text", "label"]]).map(tokenize_fn, batched=True)

print("Tokenization complete!")

### 5. Initialize Model & Evaluation Metrics

In [ ]:
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(categories),
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")
    return {"accuracy": acc, "macro_f1": macro_f1}

print("Model initialized with classification head for", len(categories), "categories.")

### 6. Fine-Tune the Transformer Model

In [ ]:
from transformers import TrainingArguments, Trainer
import inspect

# Check version compatibility for eval_strategy / evaluation_strategy
eval_kwarg = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters else "evaluation_strategy"

training_kwargs = {
    "output_dir": "./distilbert-complaints-checkpoints",
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "num_train_epochs": 3,
    "weight_decay": 0.01,
    eval_kwarg: "epoch",
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1",
    "fp16": torch.cuda.is_available(),
    "logging_steps": 50,
    "report_to": "none"
}

training_args = TrainingArguments(**training_kwargs)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": val_dataset,
    "compute_metrics": compute_metrics,
}
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print("Starting training loop...")
trainer.train()

### 7. Evaluate on Unseen Test Tickets & Plot Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

predictions_raw = trainer.predict(test_dataset)
y_pred = np.argmax(predictions_raw.predictions, axis=-1)
y_true = test_df["label"].values

print("=== Fine-Tuned DistilBERT Test Performance ===\n")
print(classification_report(y_true, y_pred, target_names=categories))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=categories, yticklabels=categories)
plt.xlabel("Predicted Department")
plt.ylabel("Actual Department")
plt.title("Confusion Matrix - Fine-Tuned DistilBERT")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 8. Test Live Inference on Custom Complaints

In [ ]:
import torch.nn.functional as F

def test_complaint(text, top_k=3):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()

    top_idx = np.argsort(probs)[::-1][:top_k]
    print(f"\nComplaint: \"{text}\"")
    for i in top_idx:
        print(f"  → {id2label[i]}: {probs[i]*100:.2f}%")

test_complaint("My flight from Dallas was delayed for 6 hours and my bags are lost.")
test_complaint("My debit card was charged an unauthorized $80 fee by the bank.")
test_complaint("I ordered a television on prime and it was delivered with a broken screen.")
test_complaint("My phone screen turned completely black after installing the latest update.")

### 9. Export Model Weights & Download Zip

In [ ]:
EXPORT_DIR = "./fine_tuned_distilbert_complaints"
model.save_pretrained(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)
print(f"Model and tokenizer saved to: {EXPORT_DIR}")

if IN_COLAB:
    !zip -r fine_tuned_distilbert.zip ./fine_tuned_distilbert_complaints
    print("\nDone! Download fine_tuned_distilbert.zip from the Colab file browser.")